<a href="https://colab.research.google.com/github/ravindyaparami/multi-agents-on-a-lie-group/blob/main/Drake%5CSingle_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

class BodyConfig:
    """
    Configuration for a general rigid body.
    Defined purely by mass and inertia tensor.
    """

    def __init__(self,
                 mass: float,
                 inertia_matrix: np.ndarray,
                 initial_position=None,
                 initial_orientation=None,
                 initial_angular_velocity=None,
                 initial_linear_velocity=None,
                 gravity=None,
                 friction=(0.9, 0.8)
                 ):

        # ---- Fundamental physical properties ----
        self.mass = float(mass)
        self.inertia_matrix = np.array(inertia_matrix, dtype=float)

        # ---- Initial position ----
        self.initial_position = (
            np.array(initial_position, dtype=float)
            if initial_position is not None
            else np.array([0.0, 0.0, 1.0])
        )

        # Orientation stored as rotation matrix (3x3)
        self.initial_orientation = (
            np.array(initial_orientation, dtype=float)
            if initial_orientation is not None
            else np.eye(3)
        )

        # ---- Initial Angular velocity ----
        self.initial_angular_velocity = (
            np.array(initial_angular_velocity, dtype=float)
            if initial_angular_velocity is not None
            else np.zeros(3)
        )

        # ---- Initial Linear velocity ----
        self.initial_linear_velocity = (
            np.array(initial_linear_velocity, dtype=float)
            if initial_linear_velocity is not None
            else np.zeros(3)
        )

        #---- Optional simulation properties ----
        self.gravity = (
            np.array(gravity, dtype=float)
            if gravity is not None
            else np.array([0.0, 0.0, -9.81])
        )

        self.friction = friction

        # ---- Validate physical correctness ----
        self._validate()

    def _validate(self):
        if self.mass <= 0:
            raise ValueError("Mass must be positive.")

        if self.inertia_matrix.shape != (3, 3):
            raise ValueError("Inertia matrix must be 3x3.")

        # Must be symmetric
        if not np.allclose(self.inertia_matrix,
                           self.inertia_matrix.T):
            raise ValueError("Inertia matrix must be symmetric.")

        # Must be positive definite
        eigenvalues = np.linalg.eigvals(self.inertia_matrix)
        if not np.all(eigenvalues > 0):
            raise ValueError(
                "Inertia matrix must be positive definite."
            )

        # ---- Rotation matrix validity ----
        R = self.initial_orientation

        if R.shape != (3, 3):
            raise ValueError("Initial orientation must be 3x3.")

        # Check orthogonality: R^T R ≈ I
        if not np.allclose(R.T @ R, np.eye(3), atol=1e-6):
            raise ValueError("Orientation matrix must be orthogonal (R^T R = I).")

        # Check determinant: det(R) ≈ 1
        det_R = np.linalg.det(R)
        if not np.isclose(det_R, 1.0, atol=1e-6):
            raise ValueError("Orientation matrix must have determinant +1.")

        # ---- Initial Angular and Linear Velocities validity ----
        if self.initial_linear_velocity.shape != (3,):
            raise ValueError("Initial linear velocity must be a 3D vector.")

        if self.initial_angular_velocity.shape != (3,):
            raise ValueError("Initial angular velocity must be a 3D vector.")



In [ ]:
from pydrake.all import LeafSystem, Value, ExternallyAppliedSpatialForce, SpatialForce
import numpy as np


class ExternalForceSystem(LeafSystem):
    def __init__(self, plant, body, force_models):
        super().__init__()
        self.plant = plant
        self.body = body
        self.force_models = force_models

        # Declare an input port to receive the plant's state
        self.DeclareVectorInputPort(
            "plant_state",
            plant.num_positions() + plant.num_velocities()
        )

        self.DeclareAbstractOutputPort(
            "spatial_forces",
            lambda: Value([ExternallyAppliedSpatialForce()]),
            self.CalcOutput
        )

    def CalcOutput(self, context, output):
        forces = []

        # Read state directly from the input port
        state = self.EvalVectorInput(context, 0).value()
        nq = self.plant.num_positions()
        q = state[:nq]
        v = state[nq:]
        t = context.get_time()

        for model in self.force_models:
            result = model(t, q, v)

            # IMPORTANT:
            # Force models must return:
            # f_vec  -> expressed in WORLD frame
            # tau_vec -> expressed in WORLD frame
            # p_vec  -> expressed in BODY frame

            if not isinstance(result, (tuple, list)):
                raise ValueError("Force model must return tuple or list")

            # ---- Allow multiple return formats ----
            if len(result) == 1:          #Model has force only, no tau, and force act through COM
                f_vec = result[0]
                tau_vec = np.zeros(3)
                p_vec = np.zeros(3)

            elif len(result) == 2:        #Model has force and tau, and force act through COM
                f_vec, tau_vec = result
                p_vec = np.zeros(3)

            elif len(result) == 3:        #Model has force and tau, and force act through different point than COM
                f_vec, tau_vec, p_vec = result

            else:
                raise ValueError("Force model must return (f), (f,tau), or (f,tau,p)")

            if f_vec.shape != (3,) or tau_vec.shape != (3,) or p_vec.shape != (3,):
                raise ValueError("f, tau, p must be 3D vectors")


            force = ExternallyAppliedSpatialForce()   #wrench
            force.body_index = self.body.index()

            # ---- CHANGE: allow custom application point ----
            force.p_BoBq_B = p_vec      #Position vector from body origin Bo to the point of application Bq, expressed in the body frame

            # ---- Include torque and force ----
            force.F_Bq_W = SpatialForce(tau=tau_vec, f=f_vec)
            forces.append(force)

        output.set_value(forces)

In [ ]:
from pydrake.all import *
import numpy as np
from pydrake.all import SpatialVelocity

class RigidBodySimulator:
    """
    Wrapper around Drake to simulate a single rigid body
    with customizable physics.
    """
    def __init__(self, config: BodyConfig, force_models = None, time_step: float = 0.001):
        self.config = config
        self.force_models = force_models
        self.time_step = time_step
        self._build_system()

    def _build_system(self):
        # 1. Diagram & plant
        self.builder = DiagramBuilder()
        self.plant, self.scene_graph = AddMultibodyPlantSceneGraph(
            self.builder,
            MultibodyPlant(time_step=self.time_step)
        )

        # 2. Add body
        self._add_body()

        # 3. Add ground
        self._add_ground()

        # 4. Finalize plant
        self.plant.Finalize()

        # 4.1 Add external force system if provided
        if self.force_models:
            self.force_system = ExternalForceSystem(
                self.plant,
                self.body,
                self.force_models
            )

            self.builder.AddSystem(self.force_system)

            self.builder.Connect(
                self.plant.get_state_output_port(),
                self.force_system.get_input_port(0)
            )

            self.builder.Connect(
                self.force_system.get_output_port(),
                self.plant.get_applied_spatial_force_input_port()
            )


        # 5. Meshcat visualizer (optional, for debugging)
        self.meshcat = StartMeshcat()
        MeshcatVisualizer.AddToBuilder(
            self.builder,
            self.scene_graph,
            self.meshcat
        )

        # 6. Build diagram
        self.diagram = self.builder.Build()
        self.simulator = Simulator(self.diagram)

        # ---- SET INITIAL VELOCITIES ----
        context = self.simulator.get_mutable_context()
        plant_context = self.plant.GetMyContextFromRoot(context)

        V_WB = SpatialVelocity(
            w=self.config.initial_angular_velocity,
            v=self.config.initial_linear_velocity
        )

        self.plant.SetFreeBodySpatialVelocity(
            self.body,
            V_WB,
            plant_context
        )

    def _add_body(self):
        cfg = self.config

        # Convert inertia matrix to UnitInertia
        I = cfg.inertia_matrix
        m = cfg.mass

        unit_inertia = UnitInertia(
            Ixx=I[0, 0]/m,
            Iyy=I[1, 1]/m,
            Izz=I[2, 2]/m,
            Ixy=I[0, 1]/m,
            Ixz=I[0, 2]/m,
            Iyz=I[1, 2]/m
        )

        spatial_inertia = SpatialInertia(
            mass=cfg.mass,
            p_PScm_E=np.zeros(3),
            G_SP_E=unit_inertia
        )

        self.body = self.plant.AddRigidBody("body", spatial_inertia)

        # Set initial pose
        X_WB = RigidTransform(
            RotationMatrix(cfg.initial_orientation),
            cfg.initial_position
        )

        self.plant.SetDefaultFloatingBaseBodyPose(
            self.body,
            X_WB
        )

        # Minimal placeholder collision geometry
        box_size = [0.2, 0.2, 0.2]
        collision_shape = Box(box_size[0], box_size[1], box_size[2])
        self.plant.RegisterCollisionGeometry(
            self.body,
            RigidTransform(),
            collision_shape,
            "body_collision",
            CoulombFriction(*cfg.friction)
        )

        # Minimal visual geometry (for Meshcat visualization)
        visual_shape = Box(box_size[0], box_size[1], box_size[2])
        self.plant.RegisterVisualGeometry(
            self.body,
            RigidTransform(),
            visual_shape,
            "body_visual",
            [0.2, 0.6, 1.0, 1.0]  # RGBA color
        )

        # Gravityk
        self.plant.mutable_gravity_field().set_gravity_vector(cfg.gravity)

    def _add_ground(self):
        ground_shape = HalfSpace()
        X_WG = RigidTransform(RollPitchYaw(np.pi, 0, 0), [0, 0, 0])

        self.plant.RegisterCollisionGeometry(
            self.plant.world_body(),
            X_WG,
            ground_shape,
            "ground_collision",
            CoulombFriction(0.9, 0.8)
        )

        self.plant.RegisterVisualGeometry(
            self.plant.world_body(),
            X_WG,
            ground_shape,
            "ground_visual",
            [0.5, 0.5, 0.5, 1.0]
        )

    def simulate(self, duration: float = 5.0, realtime_rate: float = 1.0):
        self.simulator.set_target_realtime_rate(realtime_rate)
        self.simulator.Initialize()
        self.simulator.AdvanceTo(duration)

    def get_web_url(self) -> str:
        return self.meshcat.web_url()

    def get_state(self):
        context = self.simulator.get_context()
        plant_context = self.plant.GetMyContextFromRoot(context)
        q = self.plant.GetPositions(plant_context)
        v = self.plant.GetVelocities(plant_context)
        return q, v

In [ ]:
'''
def rotate_only(t, q, v):
    return (np.zeros(3), np.array([0, 0, 2.0]))

def force_and_spin(t, q, v):
    return (
        np.array([5.0, 0, 0]),   # force
        np.array([0, 0, 1.0])    # torque
    )

def off_center_force(t, q, v):
    return (
        np.array([5.0, 0, 0]),   # force
        np.zeros(3),             # no direct torque
        np.array([0, 0.2, 0])    # offset → creates torque automatically
    )
'''

In [ ]:
config = BodyConfig(
    mass=2.0,                             # 2 kg
    inertia_matrix=np.eye(3),              # simple inertia (body frame)
    initial_position=[1, 2, 1],            # start at origin (body frame)
    initial_linear_velocity=[0, 0, 0],     # start from rest ()
    initial_angular_velocity=[0, 0, 0],
    gravity=[0, 0, 0]                      # 🔥 disable gravity
)



In [ ]:
def test(t, q, v):
    f = np.array([0.0, 0.0, 0.0])  # 10 N in +X (world frame)
    return (f,)


In [ ]:
def test(t, q, v):
    tau = np.array([0.0, 0.0, 2.0])  # torque around Z-axis
    f = np.zeros(3)                  # no force
    return (f, tau)

In [ ]:
def test(t, q, v):
    F = np.array([10.0, 0.0, 0.0])   # force in +X
    tau = np.array([0.0, 0.0, 5.0])  # torque about Z-axis
    return (F, tau)

In [ ]:
def test(t, q, v):
    F = np.array([10.0, 0.0, 0.0])  # force in +X

    r = np.array([0.0, 0.0, 0.2])   # 1m above COM in Z

    return (F, np.zeros(3), r)

In [ ]:
def force_model_1(t, q, v):
    F = np.array([10.0, 0.0, 0.0])
    tau = np.zeros(3)
    return (F, tau)

def force_model_2(t, q, v):
    F = np.array([0.0, 5.0, 0.0])     # +Y force
    tau = np.zeros(3)
    r = np.array([0.0, 0.0, 0.2])     # above COM
    return (F, tau, r)

def force_model_3(t, q, v):
    F = np.zeros(3)
    tau = np.array([0.0, 0.0, 2.0])   # spin around Z
    return (F, tau)

force_models = [
    force_model_1,
    force_model_2,
    force_model_3
]

sim = RigidBodySimulator(
    config=config,
    force_models=force_models,
    time_step=0.001
)

In [ ]:
sim = RigidBodySimulator(
    config=config,
    force_models=[test],
    time_step=0.001
)


In [ ]:
sim.simulate(duration=5.0, realtime_rate=1.0)

In [ ]:
q, v = sim.get_state()

print("Final position:", q[-3:])
print("Final velocity:", v[-3:])

In [ ]:
q, v = sim.get_state()

print("Position:", q[4:])
print("Orientation (quat):", q[:3])

print("Angular velocity:", v[:3])
print("Linear velocity:", v[3:6])

In [ ]:
q, v = sim.get_state()

print("Positions (q):", q)
print("Velocities (v):", v)